In [ ]:
!pip install -q pdfplumber PyMuPDF pandas
import os
import numpy as np
import pandas as pd
import pdfplumber
import fitz

In [ ]:
from google.colab import files
uploaded= files.upload()

filename= list(uploaded.keys())[0]
print ("uploaded successfully")
os.makedirs("extract_images", exist_ok=True)

In [ ]:
extracted_text = []
all_tables=[]

with pdfplumber.open(filename) as pdf:
  for page_num, page in enumerate(pdf.pages):
    text= page.extract_text()
    if text:
      extracted_text.append({
          "page": page_num +1,
          "content": text
      })
    tables = page.extract_tables()
    for table_idx, table in enumerate(tables):
      if table:
        df_table = pd.DataFrame(table[1:], columns=table[0])
        all_tables.append({
            "page": page_num +1,
            "table_index": table_idx +1,
            "data": df_table
        })

doc = fitz.open(filename)
image_count =0
for page_num in range(len(doc)):
  page = doc[page_num]
  image_list= page.get_images(full=True)

  for img_index, img in enumerate(image_list):
    xref= img[0]
    base_image= doc.extract_image(xref)
    image_bytes= base_image["image"]
    image_ext = base_image["ext"]

    image_filename =f"extract_images/page_{page_num+1}_image_{img_index+1}.{image_ext}"
    with open(image_filename, "wb") as f:
      f.write(image_bytes)
    image_count += 1

print(f"Extraction Complete")
print(f"-Pages parsed for text: {len(extracted_text)}")
print(f"-Tables extracted: {len(all_tables)}")
print(f"-Images extracted and saved : {image_count}")

In [ ]:
import re

def clean_text(text):
  text= re.sub(r'http\S+|www\S+|https\S+', '',text, flags=re.MULTILINE)
# 2. Directly target and remove the exact header pattern (e.g., "SZS- 0102 naJ– 2.2")
  text = re.sub(r'SZS-\s*\d+\s+[a-zA-Z]+\s*[–\-\u2013]\s*\d+\.\d+', '', text, flags=re.IGNORECASE)

    # 3. Remove trailing reversed header/footer patterns ending with 'egaP'
  text = re.sub(r'.*?noitidE\s+\d+\s+egaP', '', text, flags=re.IGNORECASE)
  text = re.sub(r'\begaP\b', '', text, flags=re.IGNORECASE)

    # 4. Remove PDF font glyph artifacts like (cid:3) and normalize Turkish 'ı' to 'i'
  text = re.sub(r'\(cid:\d+\)', '', text)
  text = text.replace('\u0131', 'i')
  text= re.sub(r'\s+', ' ', text)
  text= text.strip()
  return text

cleaned_corpus = []
for item in extracted_text:
  cleaned_content= clean_text(item["content"])
  if len(cleaned_content)>20:
    cleaned_corpus.append({
        "page":item["page"],
        "cleaned_text": cleaned_content
    })

df_corpus = pd.DataFrame(cleaned_corpus)
print(f"Total clean text segements ready for trainning: {len(df_corpus)}")
display(df_corpus.head())

In [ ]:
df_corpus.to_json("cleaned_training_data.json", orient="records", indent=4)

if len(all_tables) > 0:
  all_tables[0]["data"].to_csv("extracted_table_1.csv", index=False)
  print("Exported first extracted table to 'extracted_table_1.csv")

from google.colab import files
files.download("cleaned_training_data.json")